# Module 0: LLM Basics for Commodity Analysis

## Learning Objectives

By completing this notebook, you will:
1. Understand how LLMs can extract structured data from commodity reports
2. Make API calls to OpenAI and Anthropic Claude for text analysis
3. Design effective prompts for commodity-specific information extraction
4. Parse LLM responses into structured formats (JSON, DataFrames)
5. Evaluate LLM output quality and handle errors gracefully

## Prerequisites

- Completed Module 0 Guide 1 (API Setup)
- OpenAI or Anthropic API key
- Understanding of commodity markets (oil, gas, agriculture)
- Basic Python and pandas

## Estimated Time: 60-75 minutes

---

## 1. Setup and Configuration

We'll set up connections to major LLM providers. In commodity analysis, we use LLMs to extract insights from text-heavy reports like:
- EIA Petroleum Status Reports
- USDA WASDE (World Agricultural Supply and Demand Estimates)
- OPEC Monthly Oil Market Reports

**Key Concept:** LLMs convert unstructured text into structured data that trading systems can use.

In [ ]:
# Standard library imports
import os
import json
from typing import Dict, List, Optional, Any
from datetime import datetime

# Third-party imports
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# LLM provider imports
import openai
from anthropic import Anthropic

# Configuration
load_dotenv()

# API credentials
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')

# Initialize clients
openai_client = openai.OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

if openai_client:
    print("✅ OpenAI client initialized")
if anthropic_client:
    print("✅ Anthropic client initialized")
if not (openai_client or anthropic_client):
    print("⚠️  WARNING: No LLM API keys found. Set OPENAI_API_KEY or ANTHROPIC_API_KEY in .env")

## 2. Sample Commodity Report Text

Let's work with a realistic excerpt from an EIA Petroleum Status Report. These weekly reports move oil markets every Wednesday at 10:30 AM ET.

**What traders care about:**
- Inventory changes (builds vs. draws)
- Production levels
- Demand indicators (refinery runs, product supplied)
- Comparisons to forecasts

In [ ]:
# Sample EIA report excerpt
SAMPLE_REPORT = """
Weekly Petroleum Status Report - January 24, 2024

U.S. commercial crude oil inventories (excluding those in the Strategic Petroleum 
Reserve) increased by 1.2 million barrels from the previous week. At 439.5 million 
barrels, U.S. crude oil inventories are about 2% above the five-year average for 
this time of year.

Total motor gasoline inventories decreased by 3.1 million barrels last week and are 
approximately 3% below the five-year average for this time of year. Finished gasoline 
inventories decreased while blending components inventories increased.

Distillate fuel inventories decreased by 1.8 million barrels last week and are about 
8% below the five-year average for this time of year. Propane/propylene inventories 
fell by 2.1 million barrels last week.

U.S. crude oil refinery inputs averaged 15.9 million barrels per day during the week 
ending January 19, 2024, which was 125,000 barrels per day more than the previous 
week's average. Refineries operated at 88.5% of their operable capacity last week.

Crude oil production in the United States averaged 13.3 million barrels per day, 
unchanged from the previous week. Over the past four weeks, crude oil production 
has averaged 13.2 million barrels per day.
"""

print("Sample Report:")
print(SAMPLE_REPORT)
print(f"\nCharacter count: {len(SAMPLE_REPORT)}")

## 3. Basic LLM Extraction

Let's use an LLM to extract key numbers from this report. The goal: convert free-form text into structured JSON.

**Prompt Design Principles:**
1. Be specific about the desired output format
2. Provide examples when possible
3. Request JSON for easy parsing
4. Ask for uncertainty indicators

In [ ]:
def extract_inventory_data_openai(report_text: str) -> dict:
    """
    Extract structured inventory data from EIA report using OpenAI.
    
    Parameters:
    -----------
    report_text : str
        Raw text from EIA petroleum status report
        
    Returns:
    --------
    dict
        Structured data with inventory changes and levels
    """
    prompt = f"""
You are a commodity market analyst extracting data from EIA petroleum reports.

Extract the following information from the report and return as JSON:

{{
  "crude_oil": {{
    "weekly_change_mb": <number or null>,
    "total_inventory_mb": <number or null>,
    "vs_five_year_avg_pct": <number or null>
  }},
  "gasoline": {{
    "weekly_change_mb": <number or null>,
    "vs_five_year_avg_pct": <number or null>
  }},
  "distillate": {{
    "weekly_change_mb": <number or null>,
    "vs_five_year_avg_pct": <number or null>
  }},
  "production": {{
    "current_mbd": <number or null>,
    "four_week_avg_mbd": <number or null>
  }},
  "refinery": {{
    "utilization_pct": <number or null>,
    "throughput_mbd": <number or null>
  }}
}}

Units: mb = million barrels, mbd = million barrels per day, pct = percent
Use positive numbers for inventory builds, negative for draws.
If information is not present, use null.

Report:
{report_text}
"""
    
    # Make API call
    response = openai_client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": "You are a precise data extraction system. Return only valid JSON."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,  # Deterministic output
        response_format={"type": "json_object"}  # Enforce JSON response
    )
    
    # Parse response
    result = json.loads(response.choices[0].message.content)
    return result

# Extract data if OpenAI is configured
if openai_client:
    extracted_data = extract_inventory_data_openai(SAMPLE_REPORT)
    print("Extracted Data:")
    print(json.dumps(extracted_data, indent=2))
else:
    print("⚠️  Skipping OpenAI extraction (API key not configured)")
    extracted_data = None

### 💡 Exercise 3.1: Verify Extraction Accuracy

**Task:** Manually verify that the LLM correctly extracted the numbers from the report.

**Requirements:**
1. Check crude oil weekly change (should be +1.2 million barrels)
2. Check crude oil total inventory (should be 439.5 million barrels)
3. Check distillate vs. 5-year average (should be -8%)
4. Print any discrepancies found

**Hints:**
- Re-read the sample report carefully
- Compare against the extracted_data dictionary
- Remember: draws are negative, builds are positive

In [ ]:
# YOUR CODE HERE
# ---------------
# Verify extraction accuracy





In [ ]:
# SOLUTION
# --------
if extracted_data:
    print("Verification Results:\n")
    
    # Check crude oil change
    expected_crude_change = 1.2
    actual_crude_change = extracted_data['crude_oil']['weekly_change_mb']
    if abs(actual_crude_change - expected_crude_change) < 0.01:
        print("✅ Crude oil change: CORRECT (1.2 mb)")
    else:
        print(f"❌ Crude oil change: INCORRECT (expected {expected_crude_change}, got {actual_crude_change})")
    
    # Check crude oil inventory
    expected_crude_inventory = 439.5
    actual_crude_inventory = extracted_data['crude_oil']['total_inventory_mb']
    if abs(actual_crude_inventory - expected_crude_inventory) < 0.01:
        print("✅ Crude oil inventory: CORRECT (439.5 mb)")
    else:
        print(f"❌ Crude oil inventory: INCORRECT (expected {expected_crude_inventory}, got {actual_crude_inventory})")
    
    # Check distillate vs 5-year average
    expected_distillate_vs_avg = -8
    actual_distillate_vs_avg = extracted_data['distillate']['vs_five_year_avg_pct']
    if abs(actual_distillate_vs_avg - expected_distillate_vs_avg) < 0.01:
        print("✅ Distillate vs 5-year avg: CORRECT (-8%)")
    else:
        print(f"❌ Distillate vs 5-year avg: INCORRECT (expected {expected_distillate_vs_avg}, got {actual_distillate_vs_avg})")
    
    # Check gasoline change (should be -3.1)
    expected_gas_change = -3.1
    actual_gas_change = extracted_data['gasoline']['weekly_change_mb']
    if abs(actual_gas_change - expected_gas_change) < 0.01:
        print("✅ Gasoline change: CORRECT (-3.1 mb draw)")
    else:
        print(f"❌ Gasoline change: INCORRECT (expected {expected_gas_change}, got {actual_gas_change})")
else:
    print("⚠️  No extracted data to verify")

## 4. Converting to DataFrame

Once we have structured JSON, we can easily convert it to a pandas DataFrame for analysis and visualization.

**Why DataFrames?**
- Easy to merge with other data sources
- Built-in analysis functions
- Export to CSV, Excel, databases

In [ ]:
def json_to_dataframe(extracted_data: dict, report_date: str = None) -> pd.DataFrame:
    """
    Convert extracted JSON data to flat DataFrame.
    
    Parameters:
    -----------
    extracted_data : dict
        Nested dictionary from LLM extraction
    report_date : str, optional
        Date of the report (YYYY-MM-DD)
        
    Returns:
    --------
    pd.DataFrame
        Flattened data with one row per metric
    """
    rows = []
    
    # Flatten nested structure
    for category, metrics in extracted_data.items():
        for metric, value in metrics.items():
            rows.append({
                'category': category,
                'metric': metric,
                'value': value,
                'report_date': report_date or 'unknown'
            })
    
    df = pd.DataFrame(rows)
    return df

# Convert our extracted data
if extracted_data:
    df = json_to_dataframe(extracted_data, report_date='2024-01-24')
    print("Extracted Data as DataFrame:")
    print(df)
    print(f"\nShape: {df.shape}")
else:
    print("⚠️  No data to convert")
    df = None

## 5. Using Claude for Extraction

Different LLMs have different strengths. Claude (by Anthropic) often excels at structured data extraction and following precise instructions.

**Claude Advantages:**
- Longer context windows (200K tokens)
- Strong instruction following
- Good at maintaining consistency

In [ ]:
def extract_inventory_data_claude(report_text: str) -> dict:
    """
    Extract structured inventory data using Anthropic Claude.
    
    Parameters:
    -----------
    report_text : str
        Raw text from EIA petroleum status report
        
    Returns:
    --------
    dict
        Structured data with inventory changes and levels
    """
    prompt = f"""
Extract petroleum inventory data from this EIA report. Return valid JSON only.

Format:
{{
  "crude_oil": {{"weekly_change_mb": X, "total_inventory_mb": Y, "vs_five_year_avg_pct": Z}},
  "gasoline": {{"weekly_change_mb": X, "vs_five_year_avg_pct": Z}},
  "distillate": {{"weekly_change_mb": X, "vs_five_year_avg_pct": Z}},
  "production": {{"current_mbd": X, "four_week_avg_mbd": Y}},
  "refinery": {{"utilization_pct": X, "throughput_mbd": Y}}
}}

Rules:
- Use positive numbers for inventory builds, negative for draws
- Use null if information not present
- Units: mb = million barrels, mbd = million barrels per day

Report:
{report_text}
"""
    
    # Make API call
    message = anthropic_client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=1024,
        temperature=0.0,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    
    # Extract JSON from response
    response_text = message.content[0].text
    
    # Claude sometimes wraps JSON in markdown, so clean it
    if "```json" in response_text:
        response_text = response_text.split("```json")[1].split("```")[0]
    elif "```" in response_text:
        response_text = response_text.split("```")[1].split("```")[0]
    
    result = json.loads(response_text.strip())
    return result

# Extract data if Claude is configured
if anthropic_client:
    claude_extracted_data = extract_inventory_data_claude(SAMPLE_REPORT)
    print("Claude Extracted Data:")
    print(json.dumps(claude_extracted_data, indent=2))
else:
    print("⚠️  Skipping Claude extraction (API key not configured)")
    claude_extracted_data = None

### 💡 Exercise 5.1: Compare LLM Outputs

**Task:** If both OpenAI and Claude extractions worked, compare their outputs to identify any differences.

**Requirements:**
1. Compare each numerical field between the two extractions
2. Calculate the absolute difference for each field
3. Flag any differences greater than 0.1
4. Determine which LLM was more accurate (if any)

**Expected Output:** Summary of differences and accuracy assessment

In [ ]:
# YOUR CODE HERE
# ---------------
# Compare LLM outputs





In [ ]:
# SOLUTION
# --------
if extracted_data and claude_extracted_data:
    print("LLM Output Comparison:\n")
    
    differences = []
    
    # Compare each category
    for category in extracted_data.keys():
        if category in claude_extracted_data:
            for metric in extracted_data[category].keys():
                if metric in claude_extracted_data[category]:
                    openai_val = extracted_data[category][metric]
                    claude_val = claude_extracted_data[category][metric]
                    
                    if openai_val is not None and claude_val is not None:
                        diff = abs(openai_val - claude_val)
                        if diff > 0.01:
                            differences.append({
                                'category': category,
                                'metric': metric,
                                'openai': openai_val,
                                'claude': claude_val,
                                'diff': diff
                            })
    
    if differences:
        print("⚠️  Differences found:")
        diff_df = pd.DataFrame(differences)
        print(diff_df)
    else:
        print("✅ Both LLMs produced identical results!")
else:
    print("⚠️  Cannot compare - need both extractions")

## 6. Handling Extraction Errors

Real-world LLM applications must handle failures gracefully:
- API timeouts
- Invalid JSON responses
- Missing data in reports
- Hallucinated numbers

**Best Practices:**
1. Validate extracted numbers against reasonable ranges
2. Implement retry logic for API failures
3. Log extraction confidence
4. Maintain human-in-the-loop for critical data

In [ ]:
def validate_extraction(data: dict) -> tuple[bool, List[str]]:
    """
    Validate extracted petroleum data for reasonableness.
    
    Parameters:
    -----------
    data : dict
        Extracted inventory data
        
    Returns:
    --------
    tuple[bool, List[str]]
        (is_valid, list_of_warnings)
    """
    warnings = []
    
    # Validate crude oil inventory level (typical range: 300-500 million barrels)
    crude_inventory = data.get('crude_oil', {}).get('total_inventory_mb')
    if crude_inventory is not None:
        if crude_inventory < 250 or crude_inventory > 600:
            warnings.append(f"⚠️  Crude inventory {crude_inventory} mb outside typical range (300-500)")
    
    # Validate weekly changes (typical: -10 to +10 million barrels)
    crude_change = data.get('crude_oil', {}).get('weekly_change_mb')
    if crude_change is not None:
        if abs(crude_change) > 20:
            warnings.append(f"⚠️  Crude weekly change {crude_change} mb unusually large (typical: -10 to +10)")
    
    # Validate production (typical US range: 11-14 mbd)
    production = data.get('production', {}).get('current_mbd')
    if production is not None:
        if production < 9 or production > 16:
            warnings.append(f"⚠️  Production {production} mbd outside typical US range (11-14)")
    
    # Validate refinery utilization (typical: 75-95%)
    utilization = data.get('refinery', {}).get('utilization_pct')
    if utilization is not None:
        if utilization < 60 or utilization > 100:
            warnings.append(f"⚠️  Refinery utilization {utilization}% outside typical range (75-95)")
    
    # Validate vs 5-year average (typical: -15% to +15%)
    for product in ['crude_oil', 'gasoline', 'distillate']:
        vs_avg = data.get(product, {}).get('vs_five_year_avg_pct')
        if vs_avg is not None:
            if abs(vs_avg) > 25:
                warnings.append(f"⚠️  {product} vs 5-year avg {vs_avg}% unusually extreme")
    
    is_valid = len(warnings) == 0
    return is_valid, warnings

# Validate our extracted data
if extracted_data:
    is_valid, warnings = validate_extraction(extracted_data)
    
    if is_valid:
        print("✅ Extraction passed all validation checks")
    else:
        print("⚠️  Validation warnings:")
        for warning in warnings:
            print(f"  {warning}")
else:
    print("⚠️  No data to validate")

### 💡 Exercise 6.1: Build a Robust Extraction Function

**Task:** Create a production-ready extraction function that handles errors gracefully.

**Requirements:**
1. Wrap LLM API call in try/except blocks
2. Implement retry logic (up to 3 attempts)
3. Validate extracted data
4. Return both data and extraction metadata (success, attempts, warnings)

**Hints:**
- Use time.sleep() between retries
- Log each attempt
- Return structured result with status code

In [ ]:
# YOUR CODE HERE
# ---------------
import time

def robust_extract(report_text: str, max_retries: int = 3) -> dict:
    """
    Production-ready extraction with error handling.
    
    Returns:
    --------
    dict with keys:
        - 'success': bool
        - 'data': dict or None
        - 'attempts': int
        - 'warnings': list
        - 'error': str or None
    """
    # YOUR CODE HERE
    pass





In [ ]:
# SOLUTION
# --------
import time

def robust_extract(report_text: str, max_retries: int = 3) -> dict:
    """
    Production-ready extraction with error handling.
    """
    result = {
        'success': False,
        'data': None,
        'attempts': 0,
        'warnings': [],
        'error': None
    }
    
    for attempt in range(1, max_retries + 1):
        result['attempts'] = attempt
        
        try:
            print(f"Attempt {attempt}/{max_retries}...")
            
            # Try extraction
            if openai_client:
                data = extract_inventory_data_openai(report_text)
            elif anthropic_client:
                data = extract_inventory_data_claude(report_text)
            else:
                raise Exception("No LLM client configured")
            
            # Validate
            is_valid, warnings = validate_extraction(data)
            
            result['success'] = True
            result['data'] = data
            result['warnings'] = warnings
            
            print(f"✅ Extraction successful on attempt {attempt}")
            if warnings:
                print(f"⚠️  {len(warnings)} validation warnings")
            
            return result
            
        except Exception as e:
            print(f"❌ Attempt {attempt} failed: {str(e)}")
            result['error'] = str(e)
            
            if attempt < max_retries:
                wait_time = 2 ** attempt  # Exponential backoff
                print(f"   Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
    
    print(f"❌ All {max_retries} attempts failed")
    return result

# Test robust extraction
print("Testing robust extraction:\n")
result = robust_extract(SAMPLE_REPORT)

print("\nFinal Result:")
print(f"  Success: {result['success']}")
print(f"  Attempts: {result['attempts']}")
if result['success']:
    print(f"  Data points extracted: {sum(len(v) for v in result['data'].values())}")
    print(f"  Warnings: {len(result['warnings'])}")
else:
    print(f"  Error: {result['error']}")

## 7. Batch Processing Multiple Reports

In practice, you'll process many reports. Let's build a batch processor that maintains a history of extractions.

**Use Cases:**
- Historical data backfill
- Daily automated processing
- Multi-source aggregation

In [ ]:
# Simulate multiple reports (in reality, you'd load these from files)
SAMPLE_REPORTS = [
    {
        'date': '2024-01-24',
        'text': SAMPLE_REPORT
    },
    {
        'date': '2024-01-17',
        'text': """Weekly Petroleum Status Report - January 17, 2024
        
U.S. commercial crude oil inventories decreased by 2.5 million barrels. At 438.3 million 
barrels, inventories are about 1% above the five-year average.

Total motor gasoline inventories increased by 1.2 million barrels and are approximately 
2% below the five-year average. 

Distillate fuel inventories increased by 0.8 million barrels and are about 7% below the 
five-year average.

U.S. crude oil refinery inputs averaged 15.8 million barrels per day. Refineries operated 
at 87.8% of capacity.

Crude oil production averaged 13.3 million barrels per day.
"""
    }
]

print(f"Sample batch: {len(SAMPLE_REPORTS)} reports")

In [ ]:
def batch_process_reports(reports: List[dict]) -> pd.DataFrame:
    """
    Process multiple reports and combine into single DataFrame.
    
    Parameters:
    -----------
    reports : List[dict]
        List of dicts with 'date' and 'text' keys
        
    Returns:
    --------
    pd.DataFrame
        Combined data from all reports
    """
    all_data = []
    
    for i, report in enumerate(reports, 1):
        print(f"\nProcessing report {i}/{len(reports)} ({report['date']})...")
        
        result = robust_extract(report['text'], max_retries=2)
        
        if result['success']:
            df = json_to_dataframe(result['data'], report_date=report['date'])
            all_data.append(df)
        else:
            print(f"  ❌ Failed to process report for {report['date']}")
    
    if all_data:
        combined = pd.concat(all_data, ignore_index=True)
        return combined
    else:
        return pd.DataFrame()

# Process batch
batch_df = batch_process_reports(SAMPLE_REPORTS)

if not batch_df.empty:
    print("\n" + "="*60)
    print("Batch Processing Complete")
    print("="*60)
    print(f"Total records: {len(batch_df)}")
    print(f"Date range: {batch_df['report_date'].min()} to {batch_df['report_date'].max()}")
    print(f"\nSample of combined data:")
    print(batch_df.head(10))
else:
    print("\n❌ Batch processing produced no data")

### 💡 Exercise 7.1: Time Series Analysis

**Task:** Using the batch-processed data, analyze how crude oil inventories changed between the two reports.

**Requirements:**
1. Filter for crude oil total inventory metric
2. Calculate the change between the two report dates
3. Determine if this matches the weekly change reported in the second report
4. Create a simple line chart showing inventory levels over time

**Expected Output:** Chart and verification of inventory change calculation

In [ ]:
# YOUR CODE HERE
# ---------------
import matplotlib.pyplot as plt





In [ ]:
# SOLUTION
# --------
import matplotlib.pyplot as plt

if not batch_df.empty:
    # Filter for crude oil total inventory
    crude_inventory = batch_df[
        (batch_df['category'] == 'crude_oil') & 
        (batch_df['metric'] == 'total_inventory_mb')
    ].copy()
    
    crude_inventory['report_date'] = pd.to_datetime(crude_inventory['report_date'])
    crude_inventory = crude_inventory.sort_values('report_date')
    
    print("Crude Oil Inventory Time Series:\n")
    print(crude_inventory[['report_date', 'value']])
    
    # Calculate change
    if len(crude_inventory) >= 2:
        change = crude_inventory['value'].iloc[-1] - crude_inventory['value'].iloc[-2]
        print(f"\nInventory change: {change:+.1f} million barrels")
        
        # Compare to reported weekly change
        reported_change = batch_df[
            (batch_df['category'] == 'crude_oil') & 
            (batch_df['metric'] == 'weekly_change_mb') &
            (batch_df['report_date'] == crude_inventory['report_date'].iloc[-1].strftime('%Y-%m-%d'))
        ]['value'].iloc[0]
        
        print(f"Reported weekly change: {reported_change:+.1f} million barrels")
        print(f"Difference: {abs(change - reported_change):.1f} million barrels")
    
    # Create chart
    plt.figure(figsize=(10, 6))
    plt.plot(crude_inventory['report_date'], crude_inventory['value'], 
             marker='o', linewidth=2, markersize=8, color='darkblue')
    plt.title('Crude Oil Inventory Levels', fontsize=14, fontweight='bold')
    plt.xlabel('Report Date', fontsize=12)
    plt.ylabel('Million Barrels', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  No data available for analysis")

## Summary

### Key Takeaways

1. **LLMs Excel at Unstructured-to-Structured Conversion** - Perfect for commodity reports that contain critical numbers buried in prose

2. **Prompt Design is Critical** - Clear output format specifications, examples, and constraints improve extraction quality

3. **Always Validate** - LLMs can hallucinate numbers. Use range checks and cross-references

4. **Different Models Have Different Strengths** - GPT-4 vs Claude may produce different results; test both

5. **Production Systems Need Error Handling** - Retry logic, validation, and logging are essential

6. **Structured Output Enables Analysis** - Converting to JSON/DataFrames allows standard data science workflows

### Skills Gained

✅ Designing prompts for data extraction  
✅ Using OpenAI and Anthropic APIs  
✅ Parsing and validating LLM responses  
✅ Converting JSON to pandas DataFrames  
✅ Implementing robust error handling  
✅ Batch processing multiple documents  

### What's Next

In **Module 1, Notebook 1** (`01_eia_extraction.ipynb`), we'll apply these techniques to real EIA Petroleum Status Reports, building a production pipeline that:
- Fetches PDFs automatically
- Extracts text from tables and prose
- Uses LLMs to structure the data
- Stores results in a time series database

### Additional Resources

- **OpenAI API Documentation:** https://platform.openai.com/docs/
- **Anthropic Claude Docs:** https://docs.anthropic.com/
- **Prompt Engineering Guide:** https://www.promptingguide.ai/
- **EIA Reports:** https://www.eia.gov/petroleum/supply/weekly/

---

**Next:** [Module 1 - EIA Report Extraction](../../module_01_report_processing/notebooks/01_eia_extraction.ipynb)